<a href="https://colab.research.google.com/github/Fraanas/Big-Data/blob/main/SG_BigData_01_spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Big Data — zajęcia 1
## Wprowadzenie do Big Data i Apache Spark

**Prowadzący:** dr inż. Szczepan Górtowski  
**Tryb pracy:** teoria → przykłady → praca własna studenta

### Cele zajęć
Po zajęciach student:
1. rozumie, czym jest Big Data i jakie problemy rozwiązuje,
2. zna miejsce Apache Spark w ekosystemie narzędzi Big Data,
3. potrafi uruchomić środowisko Spark w Google Colab,
4. potrafi wykonać podstawowe operacje przetwarzania danych w Sparku,
5. potrafi samodzielnie zrealizować proste zadanie przetwarzania danych.

## Plan zajęć

### Część 1. Wprowadzenie teoretyczne
- definicje Big Data,
- model 5V,
- ekosystem Hadoop / Spark,
- kiedy Spark ma przewagę nad klasycznym przetwarzaniem lokalnym.

### Część 2. Pokazanie przykładów zastosowania
- uruchomienie środowiska Spark,
- operacje na RDD,
- operacje na DataFrame,
- proste zapytania analityczne i agregacje.

### Część 3. Praca własna studenta
- samodzielne przetworzenie danych sprzedażowych,
- wyciągnięcie prostych insightów,
- zapis wyniku do formatu Parquet.

# Część 1. Wprowadzenie teoretyczne

## 1.1. Czym jest Big Data?
Big Data to nie tylko „dużo danych”. To sytuacja, w której skala, szybkość napływu albo różnorodność danych powodują,
że klasyczne narzędzia przestają wystarczać i trzeba sięgnąć po rozwiązania rozproszone.

## 1.2. Model 5V
- **Volume** — duża objętość danych,
- **Velocity** — szybki napływ danych,
- **Variety** — wiele formatów i źródeł,
- **Veracity** — jakość i wiarygodność danych,
- **Value** — wartość biznesowa lub badawcza.

## 1.3. Gdzie w tym wszystkim jest Spark?
Apache Spark to silnik przetwarzania rozproszonego. Pozwala wykonywać obliczenia szybciej niż klasyczny MapReduce
w wielu scenariuszach, szczególnie gdy:
- przetwarzanie jest iteracyjne,
- liczy się praca w pamięci,
- chcemy używać wygodnych interfejsów DataFrame / SQL,
- potrzebujemy jednego środowiska do batch, SQL, ML i streamingu.

## 1.4. Typowe zastosowania Sparka
- analiza logów i zdarzeń,
- agregacje sprzedażowe i finansowe,
- ETL / ELT,
- budowa cech dla modeli ML,
- przetwarzanie dużych plików tekstowych,
- analiza danych IoT lub clickstream.

## 1.5. Pytania kontrolne
Odpowiedz krótko przed przejściem dalej:
1. Dlaczego samo posiadanie dużego pliku nie zawsze oznacza Big Data?
2. Które z „5V” jest najważniejsze w danych streamingowych?
3. Dlaczego Spark dobrze sprawdza się w analizie iteracyjnej?

# Konfiguracja środowiska Spark w Google Colab

> Jeżeli pracujesz lokalnie i masz już skonfigurowane środowisko PySpark, tę sekcję możesz pominąć.
> Poniższe komórki bazują na szablonie kursu i zostały uporządkowane do pierwszych zajęć laboratoryjnych.

In [ ]:
SPARK_VERSION = "3.5.8"

!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq
!pip -q uninstall -y dataproc-spark-connect || true
!pip -q install pyspark==3.5.8

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("BigData").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print(spark.version)
!pip -q install findspark==1.3.0

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
3.5.8


In [ ]:
# Krótki test działania
spark.sparkContext.parallelize([1, 2, 3, 4, 5]).map(lambda x: x * 2).collect()

[2, 4, 6, 8, 10]

# Część 2. Pokazanie przykładów zastosowania

## Przykład A — proste przetwarzanie tekstu
Najpierw pokażemy klasyczny przykład typu **word count**. To dobry punkt wejścia do myślenia o przetwarzaniu rozproszonym:
1. dzielimy dane na mniejsze elementy,
2. mapujemy,
3. agregujemy,
4. sortujemy wynik.

In [ ]:
sample_text = '''
Apache Spark is a fast engine for large-scale data processing.
Spark supports SQL, machine learning and streaming.
Big data projects often combine storage, processing and analytics.
Processing data at scale requires reproducible pipelines.
Spark helps analysts and engineers work with distributed data.
'''.strip()

with open("sample_text.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)

text_rdd = spark.sparkContext.textFile("sample_text.txt")

word_counts = (
    text_rdd
    .flatMap(lambda line: line.lower().replace(".", "").split())
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: (-x[1], x[0]))
)

word_counts.take(10)

[('data', 4),
 ('and', 3),
 ('processing', 3),
 ('spark', 3),
 ('a', 1),
 ('analysts', 1),
 ('analytics', 1),
 ('apache', 1),
 ('at', 1),
 ('big', 1)]

### Interpretacja
Widzimy, że Spark może łatwo rozproszyć operacje typu:
- wczytanie danych,
- podział na rekordy,
- transformacje,
- agregacje.

To prosty przykład, ale ten sam schemat myślenia wykorzystuje się później przy danych sprzedażowych, logach, danych webowych czy IoT.

## Przykład B — DataFrame i podstawowe agregacje
Teraz przejdziemy do wygodniejszego, „bardziej biznesowego” sposobu pracy: **Spark DataFrame**.
Przygotujemy mały zbiór danych sprzedażowych i wykonamy kilka operacji analitycznych.

In [ ]:
import random
from datetime import datetime, timedelta
from pyspark.sql import functions as F

random.seed(42)

cities = ["Poznań", "Warszawa", "Wrocław", "Gdańsk", "Kraków"]
categories = ["Elektronika", "Dom", "Spożywcze", "Sport", "Książki"]
channels = ["online", "sklep", "mobile"]

rows = []
start = datetime(2025, 3, 1, 8, 0, 0)

for i in range(1, 121):
    category = random.choice(categories)
    city = random.choice(cities)
    channel = random.choice(channels)
    quantity = random.randint(1, 5)
    price = round(random.uniform(15, 900), 2)
    order_ts = start + timedelta(hours=random.randint(0, 240), minutes=random.randint(0, 59))
    rows.append((i, order_ts.strftime("%Y-%m-%d %H:%M:%S"), city, category, channel, quantity, price))

columns = ["order_id", "order_ts", "city", "category", "channel", "quantity", "unit_price"]

sales_df = spark.createDataFrame(rows, columns)
sales_df = (
    sales_df
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn("revenue", F.round(F.col("quantity") * F.col("unit_price"), 2))
)

sales_df.show(10, truncate=False)

+--------+-------------------+--------+-----------+-------+--------+----------+-------+
|order_id|order_ts           |city    |category   |channel|quantity|unit_price|revenue|
+--------+-------------------+--------+-----------+-------+--------+----------+-------+
|1       |2025-03-02 19:47:00|Poznań  |Elektronika|mobile |3       |231.73    |695.19 |
|2       |2025-03-01 15:05:00|Kraków  |Elektronika|online |5       |388.4     |1942.0 |
|3       |2025-03-03 10:45:00|Warszawa|Dom        |mobile |5       |38.48     |192.4  |
|4       |2025-03-09 23:55:00|Gdańsk  |Książki    |online |4       |536.5     |2146.0 |
|5       |2025-03-02 23:13:00|Warszawa|Elektronika|mobile |4       |316.12    |1264.48|
|6       |2025-03-10 08:22:00|Poznań  |Spożywcze  |online |4       |100.59    |402.36 |
|7       |2025-03-11 04:24:00|Wrocław |Książki    |online |4       |489.56    |1958.24|
|8       |2025-03-05 04:36:00|Kraków  |Elektronika|sklep  |5       |798.62    |3993.1 |
|9       |2025-03-02 04:54:00|Po

In [ ]:
sales_df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- revenue: double (nullable = true)



In [ ]:
(
    sales_df
    .groupBy("category")
    .agg(
        F.round(F.sum("revenue"), 2).alias("total_revenue"),
        F.count("*").alias("orders")
    )
    .orderBy(F.desc("total_revenue"))
    .show(truncate=False)
)

+-----------+-------------+------+
|category   |total_revenue|orders|
+-----------+-------------+------+
|Elektronika|38365.14     |24    |
|Dom        |37955.39     |24    |
|Sport      |34356.66     |25    |
|Książki    |28151.59     |22    |
|Spożywcze  |26807.77     |25    |
+-----------+-------------+------+



In [ ]:
(
    sales_df
    .groupBy("city", "channel")
    .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
    .show(truncate=False)
)

+--------+-------+-------------+
|city    |channel|total_revenue|
+--------+-------+-------------+
|Warszawa|sklep  |18582.81     |
|Poznań  |mobile |18287.96     |
|Gdańsk  |mobile |18269.42     |
|Gdańsk  |online |14853.94     |
|Gdańsk  |sklep  |13251.65     |
|Kraków  |mobile |12885.59     |
|Warszawa|online |11611.17     |
|Poznań  |sklep  |10851.65     |
|Kraków  |sklep  |9003.15      |
|Kraków  |online |8384.19      |
|Poznań  |online |7709.18      |
|Wrocław |online |7205.71      |
|Warszawa|mobile |6571.35      |
|Wrocław |sklep  |6261.61      |
|Wrocław |mobile |1907.17      |
+--------+-------+-------------+



In [ ]:
sales_df.createOrReplaceTempView("sales")

spark.sql("""
SELECT
    category,
    channel,
    ROUND(AVG(revenue), 2) AS avg_order_value,
    MAX(revenue) AS max_order_value
FROM sales
GROUP BY category, channel
ORDER BY avg_order_value DESC
""").show(truncate=False)

+-----------+-------+---------------+---------------+
|category   |channel|avg_order_value|max_order_value|
+-----------+-------+---------------+---------------+
|Dom        |sklep  |1886.64        |4261.3         |
|Dom        |online |1741.21        |4347.2         |
|Elektronika|sklep  |1707.39        |3993.1         |
|Książki    |sklep  |1645.29        |2983.12        |
|Elektronika|mobile |1613.03        |3774.4         |
|Elektronika|online |1547.12        |3463.8         |
|Sport      |mobile |1533.15        |3614.4         |
|Sport      |sklep  |1414.46        |3003.8         |
|Spożywcze  |sklep  |1349.16        |2759.5         |
|Dom        |mobile |1241.5         |4446.4         |
|Książki    |mobile |1230.29        |2526.66        |
|Sport      |online |1101.36        |2920.4         |
|Książki    |online |1054.69        |2146.0         |
|Spożywcze  |online |1025.27        |2385.72        |
|Spożywcze  |mobile |726.92         |1520.64        |
+-----------+-------+-------

In [ ]:
(
    sales_df
    .withColumn("order_hour", F.hour("order_ts"))
    .withColumn("week_day", F.date_format("order_ts", "E"))
    .filter(F.col("revenue") > 1200)
    .select("order_id", "city", "category", "channel", "order_hour", "week_day", "revenue")
    .orderBy(F.desc("revenue"))
    .show(truncate=False)
)

+--------+--------+-----------+-------+----------+--------+-------+
|order_id|city    |category   |channel|order_hour|week_day|revenue|
+--------+--------+-----------+-------+----------+--------+-------+
|88      |Gdańsk  |Dom        |mobile |16        |Fri     |4446.4 |
|113     |Gdańsk  |Dom        |online |5         |Tue     |4347.2 |
|95      |Gdańsk  |Dom        |sklep  |13        |Sat     |4261.3 |
|8       |Kraków  |Elektronika|sklep  |4         |Wed     |3993.1 |
|40      |Poznań  |Elektronika|mobile |7         |Sun     |3774.4 |
|97      |Gdańsk  |Sport      |mobile |21        |Mon     |3614.4 |
|27      |Gdańsk  |Elektronika|online |16        |Sun     |3463.8 |
|81      |Poznań  |Dom        |sklep  |22        |Fri     |3012.55|
|43      |Wrocław |Sport      |sklep  |12        |Thu     |3003.8 |
|58      |Poznań  |Elektronika|mobile |19        |Sat     |2995.88|
|114     |Gdańsk  |Książki    |sklep  |15        |Fri     |2983.12|
|39      |Warszawa|Sport      |online |11       

## Co warto zapamiętać po przykładach?
1. **RDD** pomaga zrozumieć logikę rozproszonego przetwarzania.
2. **DataFrame** jest zwykle wygodniejszy i bliższy praktyce biznesowej.
3. **Spark SQL** pozwala łączyć myślenie analityczne z mocą silnika rozproszonego.
4. W praktyce dużo pracy polega na: wczytaniu danych, czyszczeniu, agregacji, filtrowaniu i zapisie wyniku.

# Część 3. Praca własna studenta

## Zadanie: przetwarzanie danych sprzedażowych w Apache Spark
Twoim zadaniem jest samodzielnie przetworzyć przygotowany zbiór danych i wyciągnąć z niego podstawowe insighty.

Najpierw wygeneruj większy zbiór danych do ćwiczenia.

In [ ]:
import csv
import random
from datetime import datetime, timedelta

random.seed(2025)

cities = ["Poznań", "Warszawa", "Wrocław", "Gdańsk", "Kraków", "Łódź"]
categories = ["Elektronika", "Dom", "Spożywcze", "Sport", "Książki", "Moda"]
channels = ["online", "sklep", "mobile"]
payment_types = ["blik", "karta", "przelew"]
statuses = ["completed", "completed", "completed", "returned", "cancelled"]

start = datetime(2025, 4, 1, 6, 0, 0)

with open("student_sales.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["order_id", "order_ts", "city", "category", "channel", "payment_type", "quantity", "unit_price", "status"])
    for i in range(1, 1001):
        ts = start + timedelta(minutes=random.randint(0, 60 * 24 * 14))
        writer.writerow([
            i,
            ts.strftime("%Y-%m-%d %H:%M:%S"),
            random.choice(cities),
            random.choice(categories),
            random.choice(channels),
            random.choice(payment_types),
            random.randint(1, 6),
            round(random.uniform(9.99, 1500.0), 2),
            random.choice(statuses)
        ])

print("Plik student_sales.csv został utworzony.")

Plik student_sales.csv został utworzony.


In [ ]:
student_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("student_sales.csv")
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn("revenue", F.round(F.col("quantity") * F.col("unit_price"), 2))
)

student_df.show(5, truncate=False)
student_df.printSchema()

+--------+-------------------+--------+-----------+-------+------------+--------+----------+---------+-------+
|order_id|order_ts           |city    |category   |channel|payment_type|quantity|unit_price|status   |revenue|
+--------+-------------------+--------+-----------+-------+------------+--------+----------+---------+-------+
|1       |2025-04-13 22:36:00|Poznań  |Moda       |sklep  |blik        |5       |10.66     |returned |53.3   |
|2       |2025-04-14 05:08:00|Warszawa|Elektronika|sklep  |karta       |1       |74.58     |completed|74.58  |
|3       |2025-04-09 19:05:00|Poznań  |Sport      |online |przelew     |1       |835.1     |completed|835.1  |
|4       |2025-04-05 16:05:00|Warszawa|Elektronika|mobile |blik        |6       |602.42    |returned |3614.52|
|5       |2025-04-09 07:35:00|Kraków  |Elektronika|mobile |blik        |5       |305.17    |completed|1525.85|
+--------+-------------------+--------+-----------+-------+------------+--------+----------+---------+-------+
o

## Zadania do wykonania

### Zadanie 1. Kontrola jakości danych
Sprawdź:
- liczbę wierszy,
- liczbę unikalnych kategorii,
- liczbę rekordów dla każdego statusu.

### Zadanie 2. Agregacje biznesowe
Policz:
- łączny przychód tylko dla zamówień `completed`,
- przychód według kategorii,
- średnią wartość zamówienia według kanału sprzedaży.

### Zadanie 3. Analiza czasu
Dodaj kolumny:
- godzina zamówienia,
- dzień tygodnia.

Następnie wskaż:
- w których godzinach pojawia się najwyższy przychód,
- w których dniach tygodnia jest najwięcej zamówień.

### Zadanie 4. Filtrowanie i insight
Znajdź zamówienia:
- o wartości większej niż 2000,
- z kanału `mobile`,
- ze statusem `returned`.

### Zadanie 5. Zapis wyniku
Zapisz tabelę z przychodem według kategorii do formatu **Parquet**.

### Zadanie rozszerzające
Przygotuj ranking TOP 5 miast pod względem przychodu dla zamówień `completed`.

In [ ]:
# TODO: Zadanie 1
student_df.createOrReplaceTempView("students")

spark.sql("""
SELECT
  COUNT(*) AS total_rows
FROM students
""").show(truncate=False)

spark.sql("""
SELECT
  COUNT(distinct category) AS total_rows
FROM students
""").show(truncate=False)

spark.sql("""
SELECT
  status,
  COUNT(status) AS total_rows
FROM students
GROUP BY status
""").show(truncate=False)


+----------+
|total_rows|
+----------+
|1000      |
+----------+

+----------+
|total_rows|
+----------+
|6         |
+----------+

+---------+----------+
|status   |total_rows|
+---------+----------+
|completed|599       |
|cancelled|200       |
|returned |201       |
+---------+----------+



In [ ]:
# TODO: Zadanie 2
"""
łączny przychód tylko dla zamówień completed,
przychód według kategorii,
średnią wartość zamówienia według kanału sprzedaży.
"""
spark.sql("""
SELECT
  COUNT(revenue) AS total_reve
FROM students
WHERE status = 'completed'
""").show(truncate=False)

spark.sql("""
SELECT
  category,
  COUNT(revenue) AS total_reve
FROM students
GROUP BY category
""").show(truncate=False)


spark.sql("""
SELECT
  channel,
  ROUND(AVG(revenue),2) AS avg_reve
FROM students
GROUP BY channel
""").show(truncate=False)



+----------+
|total_reve|
+----------+
|599       |
+----------+

+-----------+----------+
|category   |total_reve|
+-----------+----------+
|Spożywcze  |167       |
|Książki    |166       |
|Moda       |177       |
|Elektronika|169       |
|Dom        |156       |
|Sport      |165       |
+-----------+----------+

+-------+--------+
|channel|avg_reve|
+-------+--------+
|online |2491.15 |
|mobile |2589.6  |
|sklep  |2616.21 |
+-------+--------+



In [ ]:
# TODO: Zadanie 3
"""Dodaj kolumny:
godzina zamówienia,
dzień tygodnia.
Następnie wskaż:
w których godzinach pojawia się najwyższy przychód,
w których dniach tygodnia jest najwięcej zamówień."""

spark.sql("""
SELECT
  *
FROM students
""").show(truncate=False)


spark.sql("""
SELECT
  *,
  HOUR(order_ts) AS order_hour
FROM students
""").show(truncate=False)





+--------+-------------------+--------+-----------+-------+------------+--------+----------+---------+-------+
|order_id|order_ts           |city    |category   |channel|payment_type|quantity|unit_price|status   |revenue|
+--------+-------------------+--------+-----------+-------+------------+--------+----------+---------+-------+
|1       |2025-04-13 22:36:00|Poznań  |Moda       |sklep  |blik        |5       |10.66     |returned |53.3   |
|2       |2025-04-14 05:08:00|Warszawa|Elektronika|sklep  |karta       |1       |74.58     |completed|74.58  |
|3       |2025-04-09 19:05:00|Poznań  |Sport      |online |przelew     |1       |835.1     |completed|835.1  |
|4       |2025-04-05 16:05:00|Warszawa|Elektronika|mobile |blik        |6       |602.42    |returned |3614.52|
|5       |2025-04-09 07:35:00|Kraków  |Elektronika|mobile |blik        |5       |305.17    |completed|1525.85|
|6       |2025-04-06 18:57:00|Kraków  |Elektronika|online |blik        |5       |1453.76   |completed|7268.8 |
|

In [ ]:
student_df = spark.sql("""
SELECT
  *,
  HOUR(order_ts) AS order_hour,
  DATE_FORMAT(order_ts, 'E') AS week_day
FROM students
""")

student_df.createOrReplaceTempView("students")




+--------+-------------------+--------+-----------+-------+------------+--------+----------+---------+-------+----------+--------+
|order_id|order_ts           |city    |category   |channel|payment_type|quantity|unit_price|status   |revenue|order_hour|week_day|
+--------+-------------------+--------+-----------+-------+------------+--------+----------+---------+-------+----------+--------+
|1       |2025-04-13 22:36:00|Poznań  |Moda       |sklep  |blik        |5       |10.66     |returned |53.3   |22        |Sun     |
|2       |2025-04-14 05:08:00|Warszawa|Elektronika|sklep  |karta       |1       |74.58     |completed|74.58  |5         |Mon     |
|3       |2025-04-09 19:05:00|Poznań  |Sport      |online |przelew     |1       |835.1     |completed|835.1  |19        |Wed     |
|4       |2025-04-05 16:05:00|Warszawa|Elektronika|mobile |blik        |6       |602.42    |returned |3614.52|16        |Sat     |
|5       |2025-04-09 07:35:00|Kraków  |Elektronika|mobile |blik        |5       |30

In [ ]:
spark.sql("""
SELECT
  order_hour,
  ROUND(SUM(revenue),2) AS total_reve
FROM students
GROUP BY order_hour
ORDER BY total_reve DESC
""").show(truncate=False)

spark.sql("""
SELECT
  week_day,
  ROUND(SUM(revenue),2) AS total_reve
FROM students
GROUP BY week_day
ORDER BY total_reve DESC
""").show(truncate=False)

+----------+----------+
|order_hour|total_reve|
+----------+----------+
|5         |133542.91 |
|20        |127175.72 |
|2         |125232.02 |
|12        |124884.97 |
|16        |124288.42 |
|10        |122584.82 |
|13        |120929.49 |
|18        |119211.89 |
|6         |116483.11 |
|3         |114319.8  |
|4         |110019.83 |
|21        |106688.87 |
|23        |105796.66 |
|17        |105337.57 |
|11        |104889.42 |
|1         |102537.13 |
|14        |99636.45  |
|15        |95028.21  |
|19        |91796.12  |
|8         |90930.53  |
+----------+----------+
only showing top 20 rows

+--------+----------+
|week_day|total_reve|
+--------+----------+
|Sun     |420979.95 |
|Mon     |403685.03 |
|Wed     |366653.53 |
|Tue     |361917.94 |
|Sat     |357381.27 |
|Fri     |330127.62 |
|Thu     |325043.39 |
+--------+----------+



In [ ]:
# TODO: Zadanie 4
"""
Znajdź zamówienia:
o wartości większej niż 2000,
z kanału mobile,
ze statusem returned.
"""
spark.sql("""
select *
from students
where revenue > 2000 and channel = 'mobile' and status = 'returned'
""").show(truncate=False)


+--------+-------------------+--------+-----------+-------+------------+--------+----------+--------+-------+----------+--------+
|order_id|order_ts           |city    |category   |channel|payment_type|quantity|unit_price|status  |revenue|order_hour|week_day|
+--------+-------------------+--------+-----------+-------+------------+--------+----------+--------+-------+----------+--------+
|4       |2025-04-05 16:05:00|Warszawa|Elektronika|mobile |blik        |6       |602.42    |returned|3614.52|16        |Sat     |
|67      |2025-04-09 07:14:00|Gdańsk  |Elektronika|mobile |karta       |5       |801.26    |returned|4006.3 |7         |Wed     |
|87      |2025-04-05 01:18:00|Warszawa|Spożywcze  |mobile |blik        |2       |1463.18   |returned|2926.36|1         |Sat     |
|152     |2025-04-08 12:31:00|Wrocław |Moda       |mobile |blik        |3       |816.34    |returned|2449.02|12        |Tue     |
|166     |2025-04-07 11:20:00|Łódź    |Moda       |mobile |przelew     |6       |433.05   

In [ ]:
# TODO: Zadanie 5
#Zapisz tabelę z przychodem według kategorii do formatu Parquet.

reve = spark.sql("""
SELECT *
FROM students
""")

reve.write \
    .mode("overwrite") \
    .parquet("revenue.parquet")

In [ ]:
import pandas as pd


In [ ]:
df_p = pd.read_parquet('revenue.parquet')
df_p

,order_id,order_ts,city,category,channel,payment_type,quantity,unit_price,status,revenue,order_hour,week_day
0,1,2025-04-13 22:36:00,Poznań,Moda,sklep,blik,5,10.66,returned,53.30,22,Sun
1,2,2025-04-14 05:08:00,Warszawa,Elektronika,sklep,karta,1,74.58,completed,74.58,5,Mon
2,3,2025-04-09 19:05:00,Poznań,Sport,online,przelew,1,835.10,completed,835.10,19,Wed
3,4,2025-04-05 16:05:00,Warszawa,Elektronika,mobile,blik,6,602.42,returned,3614.52,16,Sat
4,5,2025-04-09 07:35:00,Kraków,Elektronika,mobile,blik,5,305.17,completed,1525.85,7,Wed
...,...,...,...,...,...,...,...,...,...,...,...,...
995,996,2025-04-05 03:13:00,Poznań,Książki,mobile,przelew,6,1423.08,completed,8538.48,3,Sat
996,997,2025-04-09 21:51:00,Łódź,Elektronika,sklep,karta,1,830.53,completed,830.53,21,Wed
997,998,2025-04-01 09:20:00,Poznań,Moda,online,przelew,5,1300.44,completed,6502.20,9,Tue
998,999,2025-04-13 10:40:00,Gdańsk,Spożywcze,online,przelew,3,737.38,completed,2212.14,10,Sun


## Wskazówki
- Używaj funkcji z `pyspark.sql.functions`, np. `sum`, `avg`, `count`, `hour`, `date_format`.
- Najpierw przefiltruj zamówienia `completed`, jeśli analizujesz przychód.
- Zwracaj uwagę na typy kolumn po wczytaniu danych.
- Staraj się nazywać kolumny wynikowe czytelnie, np. `total_revenue`, `avg_order_value`.

## Pytania podsumowujące
1. Czym różni się podejście RDD od DataFrame?
2. Dlaczego zapis do Parquet jest korzystny w projektach danych?
3. Które operacje były najprostsze do zapisania w SQL, a które w API DataFrame?
4. Jak taki proces można rozwinąć do pełnego pipeline'u ETL?

## Praca domowa / rozwinięcie
Przerób to ćwiczenie tak, aby:
- wczytywać dane z dwóch źródeł,
- połączyć je (`join`),
- zbudować prostą tabelę wynikową dla menedżera sprzedaży,
- zapisać wynik w formacie Parquet lub CSV.